# 🔧 題目 1：零售 POS 銷售分析
# Mini Data Pipeline 工作坊

> **情境**：你是一家零售連鎖集團的資料顧問。老闆想知道哪些商品最暢銷、哪些客戶最有價值、各國市場表現如何。
>
> **Pipeline**：`CSV → pandas → SQLite (raw/cleaned/analyzed) → SQL → LLM → FastAPI → Streamlit`
>
> **資料**：[Kaggle: Online Retail II UCI](https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci)（2,000 筆取樣）
>
> 📄 詳細需求見 `requirements_spec.md`


## Section 0：環境設定

直接跑下面兩格，不需要改。


In [ ]:
import pandas as pd
import sqlite3
import os
import json

print("✅ 套件載入完成")


In [ ]:
# API Key 設定（講師會提供，沒有也能跑 fallback）
OPENAI_API_KEY = ""

if os.path.exists(".env"):
    with open(".env") as f:
        for line in f:
            if line.startswith("OPENAI_API_KEY"):
                OPENAI_API_KEY = line.strip().split("=", 1)[1]

print("✅ API Key 已設定" if OPENAI_API_KEY else "⚠️ 無 API Key，使用 fallback（不影響完成度）")


---
## Section 1：Extract — 讀取資料 + 寫入 raw 表

> **目標**：把 CSV 讀進來，寫入 SQLite 的 `raw_orders` 表。
> 這是 pipeline 第一步：**資料進入系統**。


In [ ]:
# === 讀取 CSV ===
df_raw = pd.read_csv("data/raw/topic_1/orders.csv")
print(f"📊 {len(df_raw)} 筆, {len(df_raw.columns)} 欄")
print(f"欄位: {list(df_raw.columns)}")
df_raw.head()


### 🔍 檢查資料品質

跑下面這格，觀察：
- 每個欄位是什麼型別？
- 有沒有缺漏值？
- 數值欄位的範圍合不合理？


In [ ]:
# === 檢查 ===
print("=== 型別 ===")
print(df_raw.dtypes)
print("\n=== 缺漏值 ===")
print(df_raw.isnull().sum())
print("\n=== 數值統計 ===")
print(df_raw.describe())


### 寫入 SQLite raw 表

> 💡 `df.to_sql()` 一行就能把 DataFrame 寫入資料庫。
> 之後用 `pd.read_sql()` 從資料庫查詢，而不是從 CSV 讀。


In [ ]:
# === 建立 SQLite + 寫入 raw 表 ===
DB_PATH = "pipeline.db"
conn = sqlite3.connect(DB_PATH)
df_raw.to_sql("raw_orders", conn, if_exists="replace", index=False)

# 驗證
result = pd.read_sql("SELECT COUNT(*) as total FROM raw_orders", conn)
print(f"✅ raw_orders: {result['total'][0]} 筆")


---
## Section 2：Transform — 清洗 + 寫入 cleaned 表

> **目標**：從 `raw_orders` 讀出 → 清洗 → 寫入 `cleaned_orders`。
>
> 清洗策略：
> 1. 刪除 description 或 customer_id 為空的列
> 2. invoice_date 轉成 datetime，提取年/月/星期/小時
> 3. 計算 total_amount
> 4. 過濾異常值（quantity ≤ 0 或 unit_price ≤ 0）


In [ ]:
# === 從 raw 表讀出 ===
df = pd.read_sql("SELECT * FROM raw_orders", conn)
before = len(df)
print(f"從 raw_orders 讀出 {before} 筆")


### 清洗步驟 1：處理缺漏值

> 🔍 **檢查**：哪些欄位有空值？
> 🎯 **要做**：刪除 description 或 customer_id 為空的列
> 💡 **提示**：`df.dropna(subset=["欄位1", "欄位2"])`


In [ ]:
# TODO: 刪除 description 或 customer_id 為空的列


print(f"清洗前: {before} → 清洗後: {len(df)} 筆")


### 清洗步驟 2：日期轉換 + 新增時間欄位

> 🔍 **檢查**：invoice_date 目前是什麼型別？
> 🎯 **要做**：轉成 datetime，然後提取 year, month, day_of_week, hour
> 💡 **提示**：
> - `pd.to_datetime(df["欄位"])`
> - `df["欄位"].dt.year` / `.dt.month` / `.dt.day_name()` / `.dt.hour`


In [ ]:
# TODO: invoice_date 轉 datetime


# TODO: 提取 year, month, day_of_week, hour


print(df[["invoice_date", "year", "month", "day_of_week", "hour"]].head())


### 清洗步驟 3：計算總金額 + 過濾異常值

> 🎯 **要做**：
> 1. 確認 total_amount = quantity × unit_price
> 2. 過濾掉 quantity ≤ 0 或 unit_price ≤ 0 的列
> 💡 **提示**：`df = df[(df["quantity"] > 0) & (df["unit_price"] > 0)]`


In [ ]:
# TODO: 確認 total_amount


# TODO: 過濾異常值


print(f"最終: {len(df)} 筆")
print(f"總金額範圍: {df['total_amount'].min():.2f} - {df['total_amount'].max():.2f}")


### 🏁 清洗檢查點

> 跑下面這格。全部 ✅ 才往下。


In [ ]:
# 檢查點（不需要改）
assert df.isnull().sum().sum() == 0, "❌ 還有缺漏值"
assert (df["quantity"] > 0).all(), "❌ quantity 有非正值"
assert (df["unit_price"] > 0).all(), "❌ unit_price 有非正值"
assert "year" in df.columns, "❌ 缺少 year 欄位"
assert "month" in df.columns, "❌ 缺少 month 欄位"
print("✅ 全部檢查通過！")
print(f"清洗後: {len(df)} 筆, {len(df.columns)} 欄")


In [ ]:
# 寫入 cleaned 表（不需要改）
df.to_sql("cleaned_orders", conn, if_exists="replace", index=False)
print(f"✅ cleaned_orders: {len(df)} 筆")


---
## Section 3：SQL 統計分析

> **目標**：用 SQL 從 cleaned_orders 表做統計。
>
> 要回答的問題：
> 1. 哪些商品銷售額最高？
> 2. 哪些國家貢獻最多營收？
> 3. 有沒有時間趨勢？
>
> 💡 下面第一格給你範例 SQL，第二格要你**自己寫 SQL**。


In [ ]:
# === 範例：商品銷售排行 ===
product_stats = pd.read_sql("""
SELECT description, 
       COUNT(*) as order_count,
       SUM(quantity) as total_qty,
       ROUND(SUM(total_amount), 2) as total_revenue
FROM cleaned_orders
GROUP BY description
ORDER BY total_revenue DESC
LIMIT 20
""", conn)
print("📊 商品銷售 Top 20：")
product_stats


### 🎯 你的 SQL：各國銷售統計

> 寫一個 SQL 查詢，回答：各國有多少客戶、多少訂單、多少營收？
> 💡 提示：`GROUP BY country`、`COUNT(DISTINCT customer_id)`、`SUM(total_amount)`


In [ ]:
# TODO: 寫你的 SQL
country_stats = pd.read_sql("""

""", conn)
country_stats


### 🎯 自由發揮：你想看什麼？

> 用 SQL 從 cleaned_orders 查你好奇的東西。例如：
> - 每月銷售趨勢？
> - 尖峰時段？
> - 客戶消費金額排行？


In [ ]:
# TODO: 自由發揮 — 寫你的 SQL 查詢
my_analysis = pd.read_sql("""

""", conn)
my_analysis


In [ ]:
# 存統計結果（不需要改）
os.makedirs("data/processed", exist_ok=True)
product_stats.to_csv("data/processed/product_stats.csv", index=False)
# 如果你有 country_stats，也存一份
try:
    country_stats.to_csv("data/processed/country_stats.csv", index=False)
except: pass
print("✅ 統計結果已存")


---
## Section 4：LLM 加值分析

> **目標**：用 LLM 對商品描述做品類分類，結果寫入 `analyzed_orders` 表。
>
> 下面的 helper 函式已寫好，**你要做的是**：
> 1. 跑單筆測試，看分類結果合不合理
> 2. 如果不滿意，**改 prompt 或 fallback 規則**
> 3. 跑批次分析


In [ ]:
# === LLM Helper（已寫好，可以改 prompt 或 fallback 規則）===
import requests

def llm_analyze(text, api_key=None):
    if api_key:
        return _llm_api(text, api_key)
    return _llm_fallback(text)

def _llm_api(text, api_key):
    # 🎯 你可以改這段 prompt
    prompt = f"""請分析以下零售商品描述，回傳 JSON：
{{"category": "家飾/禮品/餐具/季節商品/文具/其他", "insight": "一句話商品洞察"}}

商品描述：{text[:300]}"""
    try:
        resp = requests.post("https://api.openai.com/v1/chat/completions",
            headers={"Authorization": f"Bearer {api_key}"},
            json={"model": "gpt-4o-mini", "messages": [{"role": "user", "content": prompt}], "temperature": 0.3},
            timeout=30)
        content = resp.json()["choices"][0]["message"]["content"].strip()
        if content.startswith("```"): content = content.split("\n", 1)[1].rsplit("```", 1)[0]
        return json.loads(content)
    except:
        return _llm_fallback(text)

def _llm_fallback(text):
    # 🎯 你可以加更多關鍵字改善分類
    t = text.lower()
    if any(w in t for w in ["christmas", "xmas", "santa", "winter"]):
        cat = "季節商品"
    elif any(w in t for w in ["candle", "holder", "frame", "lamp"]):
        cat = "家飾"
    elif any(w in t for w in ["cup", "mug", "plate", "bowl"]):
        cat = "餐具"
    elif any(w in t for w in ["pen", "pencil", "notebook", "card"]):
        cat = "文具"
    elif any(w in t for w in ["gift", "bag", "box", "ribbon"]):
        cat = "禮品"
    else:
        cat = "其他"
    return {"category": cat, "insight": text[:50] + "..."}

print("✅ LLM Helper 已定義")


### ⚡ 單筆測試

> 先跑 1 筆看結果。不滿意就回去改 prompt 或 fallback 規則。


In [ ]:
# 單筆測試（不需要改）
test = df["description"].iloc[0]
print(f"📝 輸入: {test}")
result = llm_analyze(test, OPENAI_API_KEY if OPENAI_API_KEY else None)
print(f"🤖 分類: {result['category']}")
print(f"🤖 洞察: {result['insight']}")
print("\n💡 結果合理嗎？不滿意就改上面的 prompt 或 fallback 規則。")


### 📦 批次分析

> 先跑 50 筆確認品質。BATCH_SIZE 可以改大。


In [ ]:
# 批次分析（可以改 BATCH_SIZE）
BATCH_SIZE = 50
api_key = OPENAI_API_KEY if OPENAI_API_KEY else None

results = []
for i, row in df.head(BATCH_SIZE).iterrows():
    r = llm_analyze(str(row["description"]), api_key)
    results.append(r)
    if len(results) % 10 == 0: print(f"  進度: {len(results)}/{BATCH_SIZE}")

df_analyzed = df.head(BATCH_SIZE).copy()
df_analyzed["category"] = [r["category"] for r in results]
df_analyzed["llm_insight"] = [r["insight"] for r in results]

print(f"\n✅ 完成 {len(results)} 筆")
print(f"\n📊 品類分佈：\n{df_analyzed['category'].value_counts()}")


In [ ]:
# 寫入 analyzed 表（不需要改）
df_analyzed.to_sql("analyzed_orders", conn, if_exists="replace", index=False)
print("\n📊 三表狀態：")
for t in ["raw_orders", "cleaned_orders", "analyzed_orders"]:
    n = pd.read_sql(f"SELECT COUNT(*) as n FROM {t}", conn)["n"][0]
    print(f"  {t}: {n} 筆")


---
## Section 5：驗證 pipeline

> 確認三張表的資料是一致的。這就是 **data lineage** 的概念。


In [ ]:
# 跨表查詢（不需要改）
lineage = pd.read_sql("""
SELECT 'raw_orders' as layer, COUNT(*) as rows FROM raw_orders
UNION ALL SELECT 'cleaned_orders', COUNT(*) FROM cleaned_orders
UNION ALL SELECT 'analyzed_orders', COUNT(*) FROM analyzed_orders
""", conn)
print("📊 Pipeline 資料流：")
print(lineage.to_string(index=False))
print("\n💡 raw → cleaned 減少 = 清洗掉了髒資料")
print("💡 cleaned → analyzed 減少 = 只分析了前 N 筆")


---
## Section 6：產出報告

> 🎯 用下面的模板產出報告。你可以**自由修改**報告內容。


In [ ]:
# TODO: 可以自由修改報告內容
total_rev = pd.read_sql("SELECT ROUND(SUM(total_amount),2) as r FROM cleaned_orders", conn)["r"][0]
cat_dist = df_analyzed["category"].value_counts().to_dict()

report = f"""# 零售 POS 銷售分析報告

## 資料概要
- 分析筆數：{len(df)} 筆交易
- 總銷售額：${total_rev:,.2f}
- 資料來源：UCI Online Retail II

## 品類分佈（LLM 分析 {len(df_analyzed)} 筆）
{chr(10).join(f'- {cat}: {cnt} 筆' for cat, cnt in cat_dist.items())}

## 建議
（TODO: 根據你的分析結果，寫 2-3 條建議）


## Pipeline
CSV → pandas → SQLite(raw/cleaned/analyzed) → SQL → LLM → 本報告
"""

os.makedirs("output", exist_ok=True)
with open("output/report.md", "w") as f: f.write(report)
print("✅ 報告已存到 output/report.md")


---
## Section 7：打包確認


In [ ]:
# 自動檢查（不需要改）
checks = [
    ("pipeline.db", "SQLite 資料庫"),
    ("data/processed/product_stats.csv", "商品統計"),
    ("output/report.md", "顧問報告"),
]
print("📋 產出確認：")
for path, desc in checks:
    print(f"  {'✅' if os.path.exists(path) else '❌'} {desc}: {path}")

if os.path.exists("pipeline.db"):
    c = sqlite3.connect("pipeline.db")
    for t in ["raw_orders", "cleaned_orders", "analyzed_orders"]:
        try:
            n = pd.read_sql(f"SELECT COUNT(*) as n FROM {t}", c)["n"][0]
            print(f"  ✅ SQLite 表 {t}: {n} 筆")
        except: print(f"  ❌ SQLite 表 {t} 不存在")
    c.close()

print("\n📋 接下來：填 README + docs/upgrade_plan.md + 準備 3 分鐘 Demo")
print("📋 選做：跑 api.py + app.py 看 Dashboard")
